# OpenFold Documentation
## Training, Inference, Dataset Preparation, and HPC Workflow

This notebook reorganises the OpenFold project documentation into a Jupyter Notebook format. It combines explanations, terminal commands, Slurm scripts, and pipeline notes in one place for easier reading and reuse.

## 1. Project Overview

This project uses OpenFold to study, reproduce, and extend AlphaFold-style protein structure prediction on the Barkla2 HPC cluster.  
The workflow spans dataset preparation, model training, checkpoint resume, inference, relaxation, and structure comparison.

The work is divided into two major parts:

1. **Training pipeline**
   - preparing OpenFold-compatible PDB training data
   - downloading and organising alignments and structures
   - building alignment databases and caches
   - training a custom OpenFold model
   - resuming training from checkpoints

2. **Inference and evaluation pipeline**
   - preparing custom FASTA, mmCIF, and MSA inputs
   - running inference with three model settings
   - comparing predicted structures with ground-truth structures
   - analysing training and relaxation behaviour

The three model configurations used for comparison are:

1. AlphaFold parameter model  
2. Official OpenFold model  
3. My own trained OpenFold model

## 2. Setting Up OpenFold

This section summarises the installation and environment setup procedure used to prepare OpenFold and its dependencies before running training and inference.

### 2.1 Pre-requisites

- CUDA 12 and PyTorch 2 are required for the supported OpenFold build described here.
- All main package dependencies are listed in the OpenFold `environment.yml` file.
- At the time of writing, only Linux systems are officially supported.

### 2.2 Installation Procedure

1. Clone the OpenFold repository.  
2. Create a conda or mamba environment from `environment.yml`.  
3. Activate the environment.  
4. Run the setup script to install third-party dependencies and configure kernels.  
5. Set the library path variables so the conda environment libraries are visible.  
6. Download model parameters.  
7. Run unit tests to verify the installation.

In [ ]:
git clone https://github.com/aqlaboratory/openfold.git
cd openfold

mamba env create -n openfold_env -f environment.yml
conda activate openfold_env

scripts/install_third_party_dependencies.sh

export LIBRARY_PATH=$CONDA_PREFIX/lib:$LIBRARY_PATH
export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH

Model parameter downloads used in the official setup guide include the following.

In [ ]:
./scripts/download_alphafold_params.sh
./scripts/download_openfold_params.sh
./scripts/download_openfold_soloseq_params.sh

To verify the installation, unit tests can be run after the OpenFold and AlphaFold parameters have been downloaded into `openfold/resources` or linked there.

In [ ]:
scripts/run_unit_tests.sh
scripts/run_unit_tests.sh -v tests.test_model

### 2.3 Optional Environment-Specific Modifications

- MPI support can be added with `pip install mpi4py`
- cuEquivariance can be installed with CUDA-specific pip packages
- If AWS access is unavailable, OpenFold parameters can also be downloaded from HuggingFace or Google Drive
- A Dockerfile is available for container-based deployment

## 3. Computational Environment

The project was conducted on the Barkla2 HPC cluster at the University of Liverpool.

### 3.1 Environment Summary

- **Cluster:** Barkla2  
- **Login host:** `barklalogin1.liv.ac.uk`  
- **Main storage used:** `fastscratch` for logs and high-I/O output, `scratch` for project outputs, and `localscratch` for node-local dataset preparation and training input  
- **Environment manager:** Conda / Miniforge  
- **Conda environment:** `openfold_env`  
- **OpenFold source directory:** `/users/sgmwu14/openfold`

### 3.2 Login and Environment Activation

In [ ]:
ssh sgmwu14@barklalogin1.liv.ac.uk
source /mnt/data2/users/sgmwu14/tools/miniforge3/etc/profile.d/conda.sh
conda activate openfold_env

### 3.3 Useful Environment Variables

In [ ]:
export PYTHONPATH=/users/sgmwu14/openfold:$PYTHONPATH
export OMP_NUM_THREADS=${SLURM_CPUS_PER_TASK}
export MKL_NUM_THREADS=${SLURM_CPUS_PER_TASK}
export TRITON_CACHE_DIR=/users/sgmwu14/fastscratch/triton_cache
mkdir -p "$TRITON_CACHE_DIR"

These settings ensure that OpenFold can be imported correctly, control CPU thread usage during training and inference, and prevent Triton cache files from being written to slower network storage.

## 4. Interactive Node Usage

Interactive sessions were used to inspect large downloads, unpack large archives, debug OpenFold commands, and test the environment before submitting full Slurm jobs.

### 4.1 Long Partition Interactive Nodes

In [ ]:
srun -p long -N 1 -n 1 -w node037 --pty bash -l
srun -p long -N 1 -w node038 --pty bash -l

### 4.2 GPU Interactive Node

In [ ]:
srun -p gpu-h100 -N 1 -w gpu32 \
  --gres=gpu:h100:1 \
  --cpus-per-task=16 \
  --mem=200G \
  --pty bash -l

## 5. OpenFold Training Dataset Preparation

To train OpenFold on PDB-based protein structure data, the following components are required:

1. MSA / alignment data  
2. mmCIF structure files  
3. duplicate chain mapping  
4. cluster files  
5. training caches  

These data follow an OpenFold / OpenProteinSet-style training setup.

## 6. Downloading OpenFold Data from RODA

### 6.1 Downloading MSA Alignment Data

In [ ]:
mkdir -p alignment_data/alignment_dir_roda

aws s3 cp s3://openfold/pdb/ \
    alignment_data/alignment_dir_roda/ \
    --recursive \
    --no-sign-request

Because the dataset is large, downloads were run as Slurm jobs rather than directly on the login node.

### 6.2 Example Batch Job Submission for Downloading

In [ ]:
cd /users/sgmwu14/localscratch
conda activate openfold_env
ls download_openfold_pdb_roda.sbatch
sbatch --nodelist=node038 download_openfold_pdb_roda.sbatch

### 6.3 Monitoring Downloads

In [ ]:
watch -n 30 '
echo "=== $(date) ===";
du -sh /users/$USER/localscratch/alignment_data/alignment_dir_roda 2>/dev/null || echo "dir not ready";
echo "Files:";
find /users/$USER/localscratch/alignment_data/alignment_dir_roda -type f 2>/dev/null | wc -l
'

In practice, monitoring very large directories occasionally caused cluster-side issues, so manual checks using `du`, `find`, and `wc -l` were also used.

## 7. Downloading mmCIF Structures and Duplicate Chain Mapping

In [ ]:
mkdir pdb_data

aws s3 cp s3://openfold/pdb_mmcif.zip pdb_data/ --no-sign-request
aws s3 cp s3://openfold/duplicate_pdb_chains.txt . --no-sign-request

unzip pdb_data/pdb_mmcif.zip -d pdb_data

- `pdb_mmcif.zip` contains experimental structure files.  
- `duplicate_pdb_chains.txt` maps duplicate chains to representative chains.

## 8. Flattening the Alignment Directory Structure

In [ ]:
bash $OF_DIR/scripts/flatten_roda.sh \
    alignment_data/alignment_dir_roda \
    alignment_data/

rm -r alignment_data/alignment_dir_roda

Expected structure after flattening:

```text
alignment_data/
└── alignments/
    ├── 1abc_A/
    ├── 2xyz_B/
    └── ...
```

## 9. Building the Alignment Subset Database

### 9.1 Recreate the Alignment Database Directory

In [ ]:
rm -rf /users/sgmwu14/localscratch/alignment_subset/alignment_dbs
mkdir -p /users/sgmwu14/localscratch/alignment_subset/alignment_dbs

### 9.2 Create a Sharded Alignment Database

In [ ]:
python /users/sgmwu14/openfold/scripts/alignment_db_scripts/create_alignment_db_sharded.py \
  /users/sgmwu14/localscratch/alignment_subset/alignments \
  /users/sgmwu14/localscratch/alignment_subset/alignment_dbs \
  alignment_db \
  --n_shards 1 \
  --duplicate_chains_file /users/sgmwu14/localscratch/pdb_data/duplicate_pdb_chains.txt

## 10. Generating FASTA and Cluster Files

### 10.1 Generate FASTA from Alignment Database

In [ ]:
python /users/sgmwu14/openfold/scripts/alignment_data_to_fasta.py \
  /users/sgmwu14/localscratch/alignment_subset/all-seqs.fasta \
  --alignment_db_index /users/sgmwu14/localscratch/alignment_subset/alignment_dbs/alignment_db.index

### 10.2 Generate 40% Sequence-Identity Clusters

In [ ]:
python /users/sgmwu14/openfold/scripts/fasta_to_clusterfile.py \
  /users/sgmwu14/localscratch/alignment_subset/all-seqs.fasta \
  /users/sgmwu14/localscratch/alignment_subset/all-seqs_clusters-40.txt \
  /mnt/data2/users/sgmwu14/tools/miniforge3/envs/openfold_env/bin/mmseqs \
  --seq-id 0.4

## 11. Downloading and Generating Cache Files

### 11.1 Download Precomputed Caches

In [ ]:
cd /users/sgmwu14/localscratch
mkdir -p pdb_data

aws s3 cp s3://openfold/data_caches/ pdb_data/data_caches/ \
  --recursive \
  --no-sign-request

### 11.2 Generate mmCIF Cache Manually

In [ ]:
python /users/sgmwu14/openfold/scripts/generate_mmcif_cache.py \
  /users/sgmwu14/localscratch/pdb_data/mmcif/mmcif_files \
  /users/sgmwu14/localscratch/pdb_data/data_caches_subset/mmcif_cache.json \
  --no_workers 16

### 11.3 Generate Chain Data Cache Manually

In [ ]:
python /users/sgmwu14/openfold/scripts/generate_chain_data_cache.py \
  /users/sgmwu14/localscratch/pdb_data/mmcif/mmcif_files \
  /users/sgmwu14/localscratch/pdb_data/data_caches_subset/chain_data_cache.json \
  --cluster_file /users/sgmwu14/localscratch/alignment_subset/all-seqs_clusters-40.txt \
  --no_workers 16

These cache files are used for sample filtering and also provide template metadata and training-time dataset access support.

## 12. Compression, Transfer, and Extraction

### 12.1 Compressing Directories

In [ ]:
cd /users/sgmwu14/scratch
tar -I 'zstd -19' -cf alignment_data_node037_20260210_083640.tar.zst alignment_data

### 12.2 Copying Archives to Local Storage

In [ ]:
cd /users/sgmwu14/localscratch
mkdir -p alignment_openfold_full_20260210
cp /users/sgmwu14/scratch/alignment_data_node037_20260210_083640.tar.zst ./alignment_openfold_full_20260210/

### 12.3 Extracting on Node-Local Storage

In [ ]:
cd /users/sgmwu14/localscratch/alignment_openfold_full_20260210
tar --use-compress-program=unzstd -xf alignment_data_node037_20260210_083640.tar.zst

### 12.4 Transfer Using tar.gz through /tmp

In [ ]:
cp /users/sgmwu14/scratch/openfold_subset_data.tar.gz /tmp/
cd /tmp
tar -xzf openfold_subset_data.tar.gz -C /users/sgmwu14/localscratch

## 13. Key Paths Used in the Project

| Purpose | Path |
|---|---|
| OpenFold source code | `/users/sgmwu14/openfold` |
| FASTA input for inference | `/users/sgmwu14/scratch/openfold_inference/fasta_dir` |
| Template mmCIF for inference | `/users/sgmwu14/localscratch/pdb_data_full/mmcif_files` |
| MSA / alignments for inference | `/users/sgmwu14/localscratch/alignment_subset/alignments` |
| AlphaFold parameters | `/users/sgmwu14/openfold/openfold/resources/params` |
| OpenFold official parameters | `/users/sgmwu14/openfold/openfold/resources/openfold_params` |
| Example custom checkpoint | `/users/sgmwu14/scratch/openfold_runs/run_subset_gpu32_h100_ep15/csv_logs/version_0/checkpoints/9-100000.ckpt` |
| AlphaFold inference output | `/users/sgmwu14/scratch/openfold_inference/alphafold_output` |
| OpenFold inference output | `/users/sgmwu14/scratch/openfold_inference/openfold_output` |
| Custom model output | `/users/sgmwu14/scratch/openfold_inference/your_model_output` |

## 14. Training Pipeline

### 14.1 Main Input Paths

In [ ]:
DATA_DIR=/users/sgmwu14/localscratch
PDB_FULL=$DATA_DIR/pdb_data_full

MMCIF_DIR=$PDB_FULL/mmcif_files
CHAIN_CACHE=$PDB_FULL/chain_data_cache.json
MMCIF_CACHE=$PDB_FULL/mmcif_cache.json
OBSOLETE=$PDB_FULL/obsolete.dat

ALIGN_DB_DIR=$DATA_DIR/alignment_subset/alignment_dbs
ALIGN_INDEX=$ALIGN_DB_DIR/alignment_db.index

### 14.2 Resume Training Script Example

In [ ]:
#!/bin/bash -l
#SBATCH -J of_subset_h100_resume
#SBATCH -p gpu-h100
#SBATCH -N 1
#SBATCH -w gpu32
#SBATCH --gres=gpu:h100:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=200G
#SBATCH --time=3-00:00:00
#SBATCH --output=/users/%u/fastscratch/of_subset_h100_%j.log
#SBATCH --error=/users/%u/fastscratch/of_subset_h100_%j.err

set -euo pipefail

set +u
source /mnt/data2/users/sgmwu14/tools/miniforge3/etc/profile.d/conda.sh
conda activate openfold_env
set -u

export TRITON_CACHE_DIR=/users/sgmwu14/fastscratch/triton_cache
mkdir -p "$TRITON_CACHE_DIR"

export PYTHONPATH=/users/sgmwu14/openfold:$PYTHONPATH
export OMP_NUM_THREADS=${SLURM_CPUS_PER_TASK}
export MKL_NUM_THREADS=${SLURM_CPUS_PER_TASK}

DATA_DIR=/users/sgmwu14/localscratch
PDB_FULL=$DATA_DIR/pdb_data_full

OUT_DIR=/users/sgmwu14/scratch/openfold_runs/run_subset_gpu32_h100_ep33
mkdir -p "$OUT_DIR"

CKPT_PATH=/users/sgmwu14/scratch/openfold_runs/run_subset_gpu32_h100_ep27/csv_logs/version_0/checkpoints/26-270000.ckpt

MMCIF_DIR=$PDB_FULL/mmcif_files
CHAIN_CACHE=$PDB_FULL/chain_data_cache.json
MMCIF_CACHE=$PDB_FULL/mmcif_cache.json
OBSOLETE=$PDB_FULL/obsolete.dat

ALIGN_DB_DIR=$DATA_DIR/alignment_subset/alignment_dbs
ALIGN_INDEX=$ALIGN_DB_DIR/alignment_db.index

EXTRA_OBSOLETE=()
if [ -f "$OBSOLETE" ]; then
  EXTRA_OBSOLETE+=( --obsolete_pdbs_file_path "$OBSOLETE" )
fi

export TORCH_CUDA_ARCH_LIST="9.0"

python -u /users/sgmwu14/openfold/train_openfold.py \
  "$MMCIF_DIR" \
  "$ALIGN_DB_DIR" \
  "$MMCIF_DIR" \
  "$OUT_DIR" \
  2021-10-10 \
  --alignment_index_path "$ALIGN_INDEX" \
  --train_chain_data_cache_path "$CHAIN_CACHE" \
  --template_release_dates_cache_path "$MMCIF_CACHE" \
  --config_preset initial_training \
  --seed 42 \
  --num_nodes 1 \
  --gpus 1 \
  --precision bf16 \
  --max_epochs 33 \
  --log_every_n_steps 25 \
  --checkpoint_every_epoch \
  --resume_from_ckpt "$CKPT_PATH" \
  "${EXTRA_OBSOLETE[@]}"

### 14.3 Submitting Training Jobs

In [ ]:
cd /users/sgmwu14/scratch
sbatch openfold_subset_gpu13.sbatch
sbatch openfold_subset_gpu15.sbatch

### 14.4 Monitoring Logs

In [ ]:
tail -f /users/sgmwu14/fastscratch/of_subset_h100_2405691.log
tail -f /users/sgmwu14/fastscratch/of_subset_h100_2405691.err

## 15. Checkpoint Resume Strategy

Training was run in multiple stages, for example epoch 15, epoch 21, epoch 27, and epoch 33.

Example checkpoint path:

`/users/sgmwu14/scratch/openfold_runs/run_subset_gpu32_h100_ep27/csv_logs/version_0/checkpoints/26-270000.ckpt`

General rule:

- if resuming from epoch 26,
- and the target is epoch 33,
- then `--max_epochs` must be set to 33.

This avoids stopping immediately after resume.

## 16. Custom Inference Dataset Preparation

### 16.1 FASTA Files

Stored in:

`/users/sgmwu14/scratch/openfold_inference/fasta_dir`

Example files: `2olo.fasta`, `4bp8.fasta`, `5i6x.fasta`, `6nbf.fasta`, `6xpf.fasta`, `7dsq.fasta`, and `7evw.fasta`.

Header example:

```text
>2olo_A
SEQUENCE...
```

### 16.2 Custom mmCIF Files

Stored in:

`/users/sgmwu14/scratch/taskcif`

Example files: `2olo.cif`, `4bp8.cif`, `5i6x.cif`, `6nbf.cif`, `6xpf.cif`, `7dsq.cif`, and `7evw.cif`.

## 17. MSA Generation with ColabFold

### 17.1 ColabFold Notebook

[Open ColabFold AlphaFold2 notebook](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)

### 17.2 Workflow

1. Open the notebook.  
2. Paste the target FASTA sequence.  
3. Run ColabFold.  
4. Download the resulting zip file.  
5. Extract and reorganise the MSA outputs.

### 17.3 Typical Output Files

In [ ]:
uniref.a3m
bfd.mgnify30.metaeuk30.smag30.a3m
pdb70.m8
out.tar.gz

### 17.4 Convert into OpenFold-Style Alignment Directories

In [ ]:
/users/sgmwu14/scratch/openfold_inference/alignments_cf/2olo_A/
    bfd_uniclust_hits.a3m
    mgnify_hits.a3m
    uniref90_hits.a3m
    pdb70.m8

## 18. Cleaning ColabFold-Generated A3M Files

In [ ]:
python3 - <<'PY'
from pathlib import Path

base = Path("/users/sgmwu14/scratch/openfold_inference/alignments_cf")
allowed = set("ACDEFGHIKLMNPQRSTVWYBXZJUO-abcdefghijklmnopqrstuvwxyz")

for f in sorted(base.glob("*/*.a3m")):
    out_lines = []
    with open(f, "rb") as fh:
        data = fh.read().replace(b"\x00", b"").replace(b"\r", b"")
    text = data.decode("utf-8", errors="ignore")

    for line in text.splitlines():
        if line.startswith(">"):
            out_lines.append(line.strip())
        else:
            cleaned = "".join(ch for ch in line.strip() if ch in allowed)
            out_lines.append(cleaned)

    with open(f, "w", newline="\n") as out:
        out.write("\n".join(out_lines) + "\n")

print("Cleaned all .a3m files")
PY

This step was essential for avoiding errors such as null-byte parsing failures during inference.

## 19. Inference Pipeline

Three-model inference was run with AlphaFold parameters, the official OpenFold checkpoint, and a custom-trained checkpoint.

### 19.1 Earlier Inference Script Example

In [ ]:
#!/bin/bash -l
#SBATCH -J of_infer_10gs
#SBATCH -p gpu-h100
#SBATCH -N 1
#SBATCH -w gpu32
#SBATCH --gres=gpu:h100:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=120G
#SBATCH --time=1-00:00:00
#SBATCH --output=/users/%u/fastscratch/of_infer_10gs_%j.log
#SBATCH --error=/users/%u/fastscratch/of_infer_10gs_%j.err

set -euo pipefail

set +u
source /mnt/data2/users/sgmwu14/tools/miniforge3/etc/profile.d/conda.sh
conda activate openfold_env
set -u

export PYTHONPATH=/users/sgmwu14/openfold:$PYTHONPATH
export OMP_NUM_THREADS=${SLURM_CPUS_PER_TASK}
export MKL_NUM_THREADS=${SLURM_CPUS_PER_TASK}
export TORCH_CUDA_ARCH_LIST="9.0"

INPUT_FASTA_DIR=/users/sgmwu14/scratch/openfold_inference/fasta_dir
TEMPLATE_MMCIF_DIR=/users/sgmwu14/localscratch/pdb_data_full/mmcif_files
PRECOMPUTED_ALIGNMENTS=/users/sgmwu14/localscratch/alignment_subset/alignments

AF_OUT=/users/sgmwu14/scratch/openfold_inference/alphafold_output
OF_OUT=/users/sgmwu14/scratch/openfold_inference/openfold_output
YOUR_OUT=/users/sgmwu14/scratch/openfold_inference/your_model_output

mkdir -p "$AF_OUT" "$OF_OUT" "$YOUR_OUT"

python3 /users/sgmwu14/openfold/run_pretrained_openfold.py \
  "$INPUT_FASTA_DIR" \
  "$TEMPLATE_MMCIF_DIR" \
  --output_dir "$AF_OUT" \
  --use_precomputed_alignments "$PRECOMPUTED_ALIGNMENTS" \
  --config_preset model_1 \
  --jax_param_path /users/sgmwu14/openfold/openfold/resources/params \
  --model_device cuda:0

python3 /users/sgmwu14/openfold/run_pretrained_openfold.py \
  "$INPUT_FASTA_DIR" \
  "$TEMPLATE_MMCIF_DIR" \
  --output_dir "$OF_OUT" \
  --use_precomputed_alignments "$PRECOMPUTED_ALIGNMENTS" \
  --config_preset model_1 \
  --openfold_checkpoint_path /users/sgmwu14/openfold/openfold/resources/openfold_params/finetuning_2.pt \
  --model_device cuda:0

python3 /users/sgmwu14/openfold/run_pretrained_openfold.py \
  "$INPUT_FASTA_DIR" \
  "$TEMPLATE_MMCIF_DIR" \
  --output_dir "$YOUR_OUT" \
  --use_precomputed_alignments "$PRECOMPUTED_ALIGNMENTS" \
  --config_preset model_1 \
  --openfold_checkpoint_path /users/sgmwu14/scratch/openfold_runs/run_subset_gpu32_h100_ep15/csv_logs/version_0/checkpoints/9-100000.ckpt \
  --model_device cuda:0

## 20. Relaxation Pipeline

### 20.1 Default Behaviour

By default, `run_pretrained_openfold.py` performs one relaxation step after inference.

### 20.2 Skip Default Relaxation

Use:

```bash
--skip_relaxation
```

This option is useful when only the raw network output is required, or when relaxation is to be controlled manually.

### 20.3 Repeated Relaxation

Repeated relaxation runs were tested on the same unrelaxed structure.

**Important finding:** relaxation is deterministic for the same input structure.

Therefore, repeating relaxation multiple times on the same unrelaxed file generally produces identical outputs unless explicit perturbations are introduced beforehand.

## 21. Result Organisation

To avoid mixing experiments, outputs were stored in separate directories.

- `/users/sgmwu14/scratch/openfold_inference/alphafold_output`
- `/users/sgmwu14/scratch/openfold_inference/openfold_output`
- `/users/sgmwu14/scratch/openfold_inference/your_model_output`

Later experiments used more explicit run-specific directories, for example:

- `/users/sgmwu14/scratch/openfold_inference_ep33_rerun1`

Repeated relaxation outputs were also separated by model.

## 22. Structure Visualization and Comparison

Predicted structures were visually compared against the experimental reference structures in UCSF ChimeraX. This stage was used to support qualitative assessment of structural improvement across checkpoints and to generate publication-quality comparison panels for the dissertation, video presentation, and presentation slides.

### 22.1 Purpose

The ChimeraX workflow was designed to:

- align predicted structures to the experimental reference structure
- visualise global fold similarity across training checkpoints
- colour predictions by pLDDT using the AlphaFold confidence palette
- export high-resolution comparison figures suitable for academic presentation and later reuse in the dissertation

### 22.2 Standard Comparison Layout

For each selected protein chain, five models were loaded:

- `#1` experimental reference structure (for example, the mmCIF file)
- `#2` prediction from the earliest selected checkpoint
- `#3` prediction from an intermediate checkpoint
- `#4` prediction from a later checkpoint
- `#5` prediction from the final selected checkpoint

In the dissertation and presentation figures, a common comparison layout was used with the following progression:

- Experimental CIF
- Epoch 2
- Epoch 10
- Epoch 20
- Epoch 32

This allowed structural changes across training to be inspected under a fixed visual convention.

### 22.3 ChimeraX Alignment and Rendering Commands

The following ChimeraX commands were used to align all predicted models to the reference structure, render them in cartoon representation, colour the reference in grey, colour the predictions by pLDDT, and export a publication-quality panel:

```chimerax
matchmaker #2 to #1
matchmaker #3 to #1
matchmaker #4 to #1
matchmaker #5 to #1

hide atoms
show cartoon

color gray #1
color byattribute bfactor #2 palette alphafold range 0,100
color byattribute bfactor #3 palette alphafold range 0,100
color byattribute bfactor #4 palette alphafold range 0,100
color byattribute bfactor #5 palette alphafold range 0,100

set bgColor white
camera ortho
lighting soft
tile columns 5

save C:/Users/Administrator/Downloads/chimeraxchain/2olo_A_reference_vs_epochs.png width 12000 height 3600 supersample 5
```

### 22.4 Interpretation of the Rendering Choices

These commands were chosen for the following reasons:

- `matchmaker` ensures that all checkpoint predictions are rigidly aligned to the same experimental reference frame
- `hide atoms` and `show cartoon` simplify the visual presentation and emphasise the backbone fold rather than atomic detail
- `color gray #1` keeps the reference visually neutral
- `color byattribute bfactor ... palette alphafold range 0,100` maps the B-factor column in the OpenFold PDB output to pLDDT colours, allowing local confidence to be inspected directly
- `camera ortho` avoids perspective distortion and is preferable for figure preparation
- `lighting soft` provides a cleaner publication-style appearance
- `tile columns 5` creates a single horizontal comparison panel with a consistent order across all structures
- the `save` command exports a high-resolution figure at fixed dimensions with supersampling, which improves edge quality and suitability for later insertion into slides and the dissertation

### 22.5 Figure Export Considerations

To keep figure presentation consistent across proteins, the same rendering settings were reused whenever possible. In particular:

- the same checkpoint order was maintained across proteins unless a checkpoint lacked a valid relaxed structure
- the same pLDDT colour mapping range (`0–100`) was used for all predicted structures
- white background and orthographic projection were used throughout
- high-resolution export settings were used to preserve readability when the figures were inserted into PowerPoint slides or the dissertation

Where a selected checkpoint did not yield a valid relaxed structure, the corresponding unrelaxed structure could be used instead, but this was treated explicitly as part of the interpretation rather than hidden.

### 22.6 Role in the Overall Project

This ChimeraX stage complements the quantitative analyses reported later in the notebook. While the CSV summaries and plots describe trends in relaxation success and confidence numerically, the ChimeraX figures provide a direct structural view of how the model predictions evolve across training checkpoints. Together, the quantitative and qualitative analyses provide a more complete account of model behaviour.


In [ ]:
# ChimeraX alignment and export commands

matchmaker #2 to #1
matchmaker #3 to #1
matchmaker #4 to #1
matchmaker #5 to #1

hide atoms
show cartoon

color gray #1
color byattribute bfactor #2 palette alphafold range 0,100
color byattribute bfactor #3 palette alphafold range 0,100
color byattribute bfactor #4 palette alphafold range 0,100
color byattribute bfactor #5 palette alphafold range 0,100

set bgColor white
camera ortho
lighting soft
tile columns 5

save C:/Users/Administrator/Downloads/chimeraxchain/2olo_A_reference_vs_epochs.png width 12000 height 3600 supersample 5


## 23. Main Difficulties Encountered

### 23.1 Dataset Preparation Complexity

OpenFold requires strict consistency across:

- FASTA headers
- alignment directory names
- mmCIF filenames
- cache files

This made dataset preparation one of the most time-consuming parts of the project.

### 23.2 MSA Generation Difficulty

OpenFold itself is mainly a training and inference framework rather than a simple MSA-generation platform.  
As a result, ColabFold was used for several custom targets.

### 23.3 HPC Environment Issues

Common practical issues included:

- job scheduling limits
- node availability
- local scratch usage
- cache placement
- long training runs requiring checkpoint resume

### 23.4 Training Log Interpretation

OpenFold training uses many loss terms rather than a single scalar objective.  
Understanding training behaviour therefore required reading both the AlphaFold methodology and the OpenFold source code.

## 24. Final Summary

The full OpenFold project pipeline can be summarised as follows:

**Prepare OpenFold-compatible training data → set up the HPC environment → download and organise alignments, mmCIF files, caches, and clusters → build subset training data for experiments → train and resume a custom OpenFold model → prepare custom inference targets using FASTA, mmCIF, and ColabFold-generated MSAs → clean alignment files if needed → run three-model inference → organise and compare outputs → analyse structure quality and relaxation behaviour.**

## 25. Relaxation Failure Analysis and pLDDT Correlation

### 25.2 Relaxation Success Criterion

Relaxation success was determined using a file-based criterion implemented in `/users/sgmwu14/scratch/analyze_relax_by_files.py`.

Rather than relying only on log messages, the script recursively scanned the repeated inference root directory:

- `/users/sgmwu14/scratch/openfold_inference_runs5times`

and identified all files matching the pattern `*_unrelaxed.pdb`.

For each unrelaxed structure, the script parsed the experimental metadata directly from the file path using a regular expression. In particular, it extracted:

- `epoch`
- `repeat`
- `chain`

from paths of the form:

```text
/users/sgmwu14/scratch/openfold_inference_runs5times/epoch_<E>/your_model_epoch_<E>/repeat_<R>/predictions/<CHAIN>_model_1_unrelaxed.pdb
```

The corresponding relaxed output path was then constructed automatically by replacing the suffix `_unrelaxed.pdb` with `_relaxed.pdb`. Relaxation success was defined operationally as the existence of this corresponding relaxed file.

Thus:

- if both `*_unrelaxed.pdb` and the corresponding `*_relaxed.pdb` existed, the sample was labelled as a successful relaxation;
- if the unrelaxed file existed but the corresponding relaxed file did not exist, the sample was labelled as a relaxation failure.

This criterion was chosen because it provides a robust sample-level definition of success at scale and directly reflects whether the relaxation stage produced a usable structural output.

---

### 25.3 Relaxation Summary Table

For every detected unrelaxed prediction, the analysis script generated a sample-level record containing:

- `epoch`
- `repeat`
- `chain`
- `status`
- `relax_success`
- `relax_error`
- `unrelaxed_path`
- `relaxed_path`
- `relaxed_exists`

These records were assembled into a single sample-level table, sorted by epoch, repeat, and chain, and saved as:

- `/users/sgmwu14/scratch/relax_analysis_results_by_files/relax_sample_table.csv`

This table preserved full traceability to the original prediction files and therefore supported both aggregate statistics and sample-level inspection.

The same script then generated grouped summaries by:

- epoch
- protein chain
- repeat

For each grouping, it computed:

- `total`
- `success`
- `errors`
- `error_rate`
- `success_rate`

The aggregated summary tables were saved as:

- `summary_by_epoch.csv`
- `summary_by_chain.csv`
- `summary_by_repeat.csv`

under:

- `/users/sgmwu14/scratch/relax_analysis_results_by_files`

In addition, the script automatically generated bar plots of relaxation error rate by epoch, by chain, and by repeat, saved as:

- `error_rate_by_epoch.png`
- `error_rate_by_chain.png`
- `error_rate_by_repeat.png`

As an integrity check, the script also compared the observed number of unrelaxed samples against the theoretical total of `33 × 7 × 5 = 1155`, and issued a warning if the counts did not match.

---

### 25.4 Global pLDDT Summary Across All Repeated Inference Outputs

To obtain a complete confidence summary across all repeated inference outputs, the script `/users/sgmwu14/scratch/extract_all_plddt.py` was used to scan both relaxed and unrelaxed PDB files under:

- `/users/sgmwu14/scratch/openfold_inference_runs5times`

The script recursively collected all files matching:

- `*_unrelaxed.pdb`
- `*_relaxed.pdb`

and then parsed the sample metadata directly from the file path using a regular expression. For each PDB file, the following metadata were extracted from the path structure:

- `epoch`
- `repeat`
- `chain`
- `state` (`unrelaxed` or `relaxed`)

For each structure, pLDDT was extracted from the PDB B-factor field. Specifically, the script used the B-factor of the CA atom as the residue-level pLDDT value, and then computed chain-level summary statistics:

- `mean_plddt`
- `min_plddt`
- `max_plddt`
- `num_residues`

This provides a whole-chain confidence summary rather than a residue-wise local confidence profile.

Files containing no valid CA atoms were skipped, and a warning message was printed for inspection. All valid entries were assembled into a single summary table containing:

- `epoch`
- `repeat`
- `chain`
- `state`
- `pdb_path`
- `mean_plddt`
- `min_plddt`
- `max_plddt`
- `num_residues`

The final table was saved as:

- `/users/sgmwu14/scratch/plddt_analysis_results/all_plddt_summary.csv`

This full summary was useful for exploratory confidence analysis across both pre-relaxation and post-relaxation structures. In contrast, the downstream relaxation-failure analysis described later used only unrelaxed structures, because relaxation failure depends on the structure before Amber relaxation is applied.

---

### 25.5 Selecting Representative Relaxed Structures by pLDDT

To support qualitative figure preparation, the script `/users/sgmwu14/scratch/select_best_relaxed_by_plddt.py` was used to identify representative relaxed structures for each chain and checkpoint.

The script took as input two previously generated summary tables:

- `/users/sgmwu14/scratch/relax_analysis_results_by_files/relax_sample_table.csv`
- `/users/sgmwu14/scratch/plddt_analysis_results/unrelaxed_plddt_summary.csv`

These tables were merged using:

- `epoch`
- `repeat`
- `chain`

Although the pLDDT summary may contain both relaxed and unrelaxed entries, this script explicitly retained only the unrelaxed pLDDT values. This ensured that the ranking criterion reflected the model’s pre-relaxation confidence rather than any post-relaxation structural state.

The merged table was then filtered to retain only samples with successful relaxation (`relax_success == 1`). Among these successful samples, the script selected, for each `(epoch, chain)` pair, the repeat with the highest `mean_plddt`. This produced a table of the best available relaxed structure per epoch and chain, saved as:

- `/users/sgmwu14/scratch/best_display_selection_by_plddt/best_relaxed_file_per_epoch_chain.csv`

This table contains, for each selected sample:

- `epoch`
- `repeat`
- `chain`
- `mean_plddt`
- `relaxed_path`
- `unrelaxed_path`

To make figure preparation easier, the script also generated a second recommendation table for visual display. For each chain, it selected up to three representative checkpoints from the sorted best-file table:

- the earliest successful epoch
- a middle epoch
- the latest epoch

Duplicate selections were removed when the number of available successful epochs was small. The resulting recommendation table was saved as:

- `/users/sgmwu14/scratch/best_display_selection_by_plddt/recommended_epochs_for_display.csv`

This second table was used as a lightweight guide for selecting representative structures for ChimeraX visual comparison, while still retaining links to both relaxed and unrelaxed files.

---

### 25.6 Selecting Chains and Epochs for Qualitative Display

To support qualitative structure comparison, the script `/users/sgmwu14/scratch/find_good_chains_epochs_for_display.py` was used to identify which protein chains and checkpoints were most suitable for visual presentation.

This script took as input the sample-level relaxation summary table:

- `/users/sgmwu14/scratch/relax_analysis_results_by_files/relax_sample_table.csv`

and aggregated it at the `(epoch, chain)` level. For each epoch–chain pair, it computed:

- `total`
- `success`
- `error`
- `success_rate`

In addition, the script derived threshold-based stability indicators:

- `all5_success`: whether all 5 repeated predictions relaxed successfully
- `atleast4_success`: whether at least 4 of 5 repeats relaxed successfully
- `atleast3_success`: whether at least 3 of 5 repeats relaxed successfully

These outputs were saved as:

- `epoch_chain_relax_counts.csv`
- `epoch_chain_all5_success.csv`
- `epoch_chain_atleast4_success.csv`
- `epoch_chain_atleast3_success.csv`

under:

- `/users/sgmwu14/scratch/display_selection_analysis`

To facilitate trend analysis, the script also generated chain–epoch matrix tables showing:

- the number of successful relaxed runs per `(chain, epoch)`
- whether a `(chain, epoch)` pair achieved full `5/5` relaxation success

These were saved as:

- `chain_epoch_success_matrix.csv`
- `chain_epoch_all5_matrix.csv`

The script then produced a chain-level display priority summary. For each chain, it computed:

- `total_success_runs`
- `total_error_runs`
- `num_all5_epochs`
- `num_atleast4_epochs`
- `num_atleast3_epochs`
- `mean_success_per_epoch`

Chains were ranked primarily by the number of `5/5`-success epochs and secondarily by mean success per epoch. This yielded a practical ordering of which targets were most suitable for display, saved as:

- `chain_display_priority_summary.csv`
- `recommended_chains_for_display.csv`

In addition, the script computed, for each chain, the first epoch at which all 5 repeated predictions successfully relaxed. This was saved as:

- `chain_first_epoch_with_all5_success.csv`

This information was useful for identifying easy targets, difficult targets, and representative examples of progressive improvement across training.

Finally, two simple visual summaries were generated:

- `success_count_by_chain.png`
- `all5_count_by_chain.png`

Together, these outputs were used to decide which chains and checkpoints should be prioritised for ChimeraX-based qualitative comparison.

---

### 25.7 Integrated Analysis of Relaxation Failure and pLDDT

The script `/users/sgmwu14/scratch/analyze_error_plddt_relationship_full.py` was used as the main integration and statistical analysis step for the relaxation-failure study.

It combined information from:

- `/users/sgmwu14/scratch/plddt_analysis_results/unrelaxed_plddt_summary.csv`
- `/users/sgmwu14/scratch/relax_analysis_results_by_files/relax_sample_table.csv`
- `/users/sgmwu14/scratch/relax_analysis_results_by_files/summary_by_chain.csv`
- `/users/sgmwu14/scratch/relax_analysis_results_by_files/summary_by_epoch.csv`
- `/users/sgmwu14/scratch/relax_analysis_results_by_files/summary_by_repeat.csv`

The script first standardised the available columns and retained only the unrelaxed pLDDT entries, so that confidence was measured before Amber relaxation rather than after it. The pLDDT table and relaxation sample table were then merged at the sample level using:

- `epoch`
- `repeat`
- `chain`

with `validate="one_to_one"` to ensure that each sample had a unique match.

The merged table was saved as:

- `/users/sgmwu14/scratch/error_plddt_analysis_results/merged_error_plddt_table.csv`

From this merged table, grouped summaries were computed by:

- epoch
- chain
- repeat

For each grouping, the script calculated:

- `total`
- `errors`
- `success`
- `mean_plddt`
- `median_plddt`
- `std_plddt`
- `min_plddt`
- `max_plddt`
- `error_rate`
- `success_rate`

These summary tables were saved as:

- `summary_error_plddt_by_epoch.csv`
- `summary_error_plddt_by_chain.csv`
- `summary_error_plddt_by_repeat.csv`

The script also generated a direct comparison of pLDDT between successful and failed samples by grouping on `relax_error`. This produced:

- `summary_plddt_by_error.csv`

To support visual interpretation, the following figures were generated:

- `boxplot_plddt_success_vs_error.png`
- `mean_plddt_by_epoch.png`
- `mean_plddt_by_chain.png`
- `mean_plddt_by_repeat.png`
- `error_rate_by_epoch.png`
- `error_rate_by_chain.png`
- `error_rate_by_repeat.png`
- `scatter_plddt_vs_error.png`

These visualisations were designed to answer the main analytical questions of the study, including whether relaxation error decreases across training, whether some chains are intrinsically harder, and whether lower-confidence predictions are more likely to fail.

In addition to descriptive summaries, the script also computed correlation coefficients:

- Pearson and Spearman correlation between `epoch` and `mean_plddt`
- Pearson and Spearman correlation between `mean_plddt` and `relax_error`

The results were saved as:

- `correlation_summary.csv`

To examine the relationship more formally, the script fitted a logistic regression model of the form:

- `relax_error ~ mean_plddt + epoch + C(chain) + C(repeat)`

This model tested whether mean pLDDT remained associated with relaxation failure after accounting for checkpoint stage, target identity, and repeat effects. The fitted model summary and coefficient table were saved as:

- `logistic_regression_summary.txt`
- `logistic_regression_coefficients.csv`

Finally, the original relaxation-only summaries were copied into the same output directory to facilitate comparison between the original relaxation analysis and the merged confidence-aware analysis.

All outputs from this script were stored under:

- `/users/sgmwu14/scratch/error_plddt_analysis_results`

---


## 26. BRI-Based Geometry Analysis Workflow

In addition to the main OpenFold evaluation pipeline, a separate BRI-based geometry-analysis workflow was used to study predicted protein backbones in invariant form. This branch consisted of three steps: local package installation, invariant computation, and invariant plotting.

### 26.1 BRI Package Installation

Before running the geometry-analysis workflow, the local BRI wheel package had to be installed. The required file was:

- `/Volumes/MinhaoWu/BRI_code_1.2.2/backbone_rigid_invariant-1.2.2-py3-none-any.whl`

This package provides the `bri` module used in later steps, including the `MiniChain` class and invariant-computation functions.

A typical installation command was:

```bash
pip install /Volumes/MinhaoWu/BRI_code_1.2.2/backbone_rigid_invariant-1.2.2-py3-none-any.whl
```

The installation could be verified with:

```bash
python -c "from bri import MiniChain; print('BRI installed successfully')"
```

This step was required before any invariant computation or plotting could be performed.

### 26.2 Invariant Computation for Predicted Structures

Invariant computation was carried out using:

- `/Volumes/rdm01/GeometryOfProteins/MinhaoWu/BRI_code_1.2.2/BRI_plotting/bri_computations.py`

The purpose of this script was to convert predicted PDB structures into tabular BRI-based invariant representations. For each input PDB file, the script:

1. read the structure using `biotite`;
2. extracted atom, residue, coordinate, and occupancy information;
3. converted the structure into a `MiniChain` object;
4. computed chain-level invariants using:
   - `get_chain_invariant(True)`
   - `get_chain_invariant_BTP()`
5. merged both outputs into one table;
6. saved the result as a CSV file with suffix `_inv.csv`.

The script processed all PDB files in a target folder, for example:

- `2olo_all_pdbs`

and wrote the outputs to:

- `<root>/<target_folder>/invariants`

Multiprocessing was used so that all available CPU cores could process files in parallel. This made the invariant-generation stage efficient for folders containing multiple prediction outputs.

### 26.3 Plotting of Invariant Profiles

After invariant CSV files had been generated, they were visualised using:

- `/Volumes/MinhaoWu/BRI_code_1.2.2/BRI_plotting/plotting_invariants.ipynb`

The notebook was used to compare predicted structures with a selected reference target in invariant space. Its main plotting function loaded invariant CSV files from a protein-specific folder, extracted the required invariant columns, and overlaid them with the invariant profile of the chosen target structure.

Two types of invariant sets were used in the notebook:

- a reduced interpretable set including backbone lengths, angles, and tau-related terms;
- the broader BRI invariant column set provided by the package.

The resulting figures were saved as PNG files and used as supplementary geometry-aware visualisations. Unlike ChimeraX images, which provide three-dimensional structural views, these plots summarise backbone behaviour in invariant form.

### 26.4 Role in the Overall Workflow

This BRI-based branch served as a supplementary geometry-analysis workflow within the project. It did not replace the main quantitative evaluation based on relaxation success and pLDDT, but added an additional representation of backbone structure for comparison and interpretation.

The overall sequence was:

1. install the local `bri` package;
2. compute invariant CSV files from predicted PDB structures;
3. plot invariant profiles for target-wise comparison.

This workflow provided an additional geometry-aware perspective on the predicted structures and complemented the main OpenFold training, inference, relaxation, and ChimeraX-based comparison pipeline.